# Context Editing Middleware

Middleware and edit strategies used to manage an agent's context size by clearing older tool outputs after the conversation exceeds a configured token threshold.

The implementation is model-agnostic and supports both approximate token counting and model-provided token counting.

---

## `DEFAULT_TOOL_PLACEHOLDER`

Default text inserted in place of a cleared tool result.

```python
DEFAULT_TOOL_PLACEHOLDER = "[cleared]"
```

---

## `TokenCounter`

Callable type used to calculate the number of tokens in a sequence of messages.

```python
TokenCounter = Callable[
    [Sequence[BaseMessage]],
    int,
]
```

- Accepts a sequence of `BaseMessage` objects.
- Returns the total token count as an integer.

---

# `ContextEdit`

Protocol describing a context-editing strategy.

- Bases: `Protocol`

Any custom context-editing strategy should implement the `apply` method.

## Methods

### 1. `apply`

Applies the context-editing strategy directly to the supplied message list.

The message list is modified in place.

- **Syntax:**

  ```python
  apply(
      self,
      messages: list[AnyMessage], # Messages that may be edited
      *,
      count_tokens: TokenCounter # Function used to count message tokens
  ) -> None
  ```

---

# `ClearToolUsesEdit`

Configuration and strategy for clearing older tool outputs when the context exceeds a token limit.

- Bases: `ContextEdit`
- Type: `dataclass`

The strategy preserves a configurable number of recent tool results and replaces eligible older results with placeholder text.

## Constructor

```python
ClearToolUsesEdit(
    trigger: int = 100_000, # Token count that activates the strategy
    clear_at_least: int = 0, # Minimum number of tokens to reclaim
    keep: int = 3, # Number of recent tool results to preserve
    clear_tool_inputs: bool = False, # Whether to remove matching tool-call arguments
    exclude_tools: Sequence[str] = (), # Tool names that must not be cleared
    placeholder: str = "[cleared]" # Replacement text for cleared tool results
)
```

## Attributes

- `trigger` — Token count above which tool-result clearing is performed.
- `clear_at_least` — Minimum number of tokens the strategy should attempt to reclaim.
- `keep` — Number of the most recent tool results that must remain unchanged.
- `clear_tool_inputs` — Determines whether the arguments of the originating tool call are also cleared.
- `exclude_tools` — Sequence of tool names whose outputs must never be cleared.
- `placeholder` — Text inserted into a cleared `ToolMessage`.

## Methods

### 1. `apply`

Applies the clear-tool-uses strategy to a list of messages.

The method:

- Counts the current message tokens.
- Stops immediately when the token count is not greater than `trigger`.
- Finds eligible `ToolMessage` objects.
- Preserves the latest `keep` tool results.
- Skips excluded tools and results that were already cleared.
- Replaces cleared tool output with `placeholder`.
- Removes the tool artifact.
- Records clearing information in `response_metadata`.
- Optionally clears the matching tool-call arguments.
- Stops once `clear_at_least` tokens have been reclaimed, when configured.

- **Syntax:**

  ```python
  apply(
      self,
      messages: list[AnyMessage], # Messages edited in place
      *,
      count_tokens: TokenCounter # Function used to count tokens
  ) -> None
  ```

### 2. `_build_cleared_tool_input_message`

Creates a copy of an `AIMessage` in which the arguments for one matching tool call are replaced with an empty dictionary.

It also records the cleared tool-call ID under the message's context-editing metadata.

> This is an internal static helper method.

- **Syntax:**

  ```python
  _build_cleared_tool_input_message(
      message: AIMessage, # AI message containing the tool call
      tool_call_id: str # ID of the tool call whose arguments are cleared
  ) -> AIMessage
  ```

---

# `ContextEditingMiddleware`

Agent middleware that automatically applies context-editing strategies before a model is invoked.

- Bases: `AgentMiddleware[AgentState[ResponseT], ContextT, ResponseT]`

The middleware copies the request messages, applies each configured edit strategy, and sends the edited messages to the model handler. The original request message list is not modified.

## Constructor

```python
ContextEditingMiddleware(
    *,
    edits: Iterable[ContextEdit] | None = None, # Editing strategies to apply
    token_count_method: Literal["approximate", "model"] = "approximate"
) -> None
```

- When `edits` is `None`, the middleware uses one default `ClearToolUsesEdit()` strategy.
- `"approximate"` uses LangChain's approximate message-token counter.
- `"model"` uses the chat model's own message-token counting method.

## Attributes

- `edits` — List of context-editing strategies applied in sequence.
- `token_count_method` — Token-counting mode: `"approximate"` or `"model"`.

## Methods

### 1. `wrap_model_call`

Synchronously applies all configured context edits before calling the model handler.

When the request contains no messages, the original request is passed directly to the handler.

- **Syntax:**

  ```python
  wrap_model_call(
      self,
      request: ModelRequest[ContextT],
      handler: Callable[
          [ModelRequest[ContextT]],
          ModelResponse[ResponseT]
      ]
  ) -> ModelResponse[ResponseT] | AIMessage
  ```

- `request` — Model request containing the messages, model, tools, state, runtime, and optional system message.
- `handler` — Synchronous callback that executes the model request.
- **Returns:** The model response produced from the request containing the potentially edited messages.

### 2. `awrap_model_call`

Asynchronously applies all configured context edits before awaiting the model handler.

When the request contains no messages, the original request is passed directly to the asynchronous handler.

- **Syntax:**

  ```python
  async awrap_model_call(
      self,
      request: ModelRequest[ContextT],
      handler: Callable[
          [ModelRequest[ContextT]],
          Awaitable[ModelResponse[ResponseT]]
      ]
  ) -> ModelResponse[ResponseT] | AIMessage
  ```

- `request` — Model request containing the messages, model, tools, state, runtime, and optional system message.
- `handler` — Asynchronous callback that executes the model request.
- **Returns:** The awaited model response produced from the request containing the potentially edited messages.

---

## Exported Components

```python
__all__ = [
    "ClearToolUsesEdit",
    "ContextEditingMiddleware",
]
```